# Dynamic Time-Period Metrics in Semantic Views

**Duration:** 20-30 minutes  
**Scenario:** You need to define metrics that calculate dynamically over adjustable time periods (monthly, quarterly, yearly) in a Snowflake semantic view, without redefining the view each time.

**What you'll learn:**
1. Use **variables** to swap time grain at query time (month/quarter/year)
2. Use **window function metrics** for rolling averages and period-over-period comparisons
3. Combine both patterns for maximum flexibility
4. Understand current limitations and workarounds

**Prerequisites:** Run `setup.sql` before starting this notebook. It creates the database, sample data, and all three semantic views.

---
## Setup & Connection

Establish a Snowpark session and verify the lab objects exist.

In [ ]:
import os
import pandas as pd

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    from snowflake.snowpark import Session
    connection_params = {
        "account": os.environ.get("SNOWFLAKE_ACCOUNT"),
        "user": os.environ.get("SNOWFLAKE_USER"),
        "password": os.environ.get("SNOWFLAKE_PASSWORD"),
        "role": os.environ.get("SNOWFLAKE_ROLE", "ACCOUNTADMIN"),
        "warehouse": "DYNAMIC_METRICS_WH",
        "database": "DYNAMIC_METRICS_DEMO",
        "schema": "PUBLIC",
    }
    session = Session.builder.configs(connection_params).create()

session.sql("USE DATABASE DYNAMIC_METRICS_DEMO").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE DYNAMIC_METRICS_WH").collect()
print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Database: DYNAMIC_METRICS_DEMO")

In [ ]:
# Verify sample data and semantic views exist
row_count = session.sql("SELECT COUNT(*) AS cnt FROM daily_revenue").collect()[0][0]
print(f"daily_revenue table: {row_count:,} rows")

views = session.sql("SHOW SEMANTIC VIEWS IN SCHEMA DYNAMIC_METRICS_DEMO.PUBLIC").to_pandas()
print(f"\nSemantic views created: {len(views)}")
print(views[['"name"', '"comment"']].to_string(index=False))

---
## Pattern 1: Variables for Swappable Time Grain

The simplest approach: define a **variable** in your semantic view that controls how the time dimension is computed. Users pass a different value at query time to switch between monthly, quarterly, or yearly aggregation.

### How it works

The semantic view `sv_variable_time_grain` defines:
- A `time_grain` variable (VARCHAR, default = `'month'`)
- A `time_period` dimension that uses a CASE expression on the variable
- Standard aggregation metrics (SUM, AVG, COUNT)

```sql
VARIABLES (
    time_grain VARCHAR DEFAULT 'month'
)
DIMENSIONS (
    revenue.time_period AS
      CASE time_grain
        WHEN 'month'   THEN TO_CHAR(sale_date, 'YYYY-MM')
        WHEN 'quarter' THEN TO_CHAR(sale_date, 'YYYY') || '-Q' || TO_CHAR(QUARTER(sale_date))
        WHEN 'year'    THEN TO_CHAR(sale_date, 'YYYY')
      END
)
```

The key insight: the **same query** returns different granularities just by changing the `VARIABLES` clause.

In [ ]:
# Monthly aggregation (default)
df = session.sql("""
SELECT * FROM SEMANTIC_VIEW(
  sv_variable_time_grain
  DIMENSIONS revenue.time_period
  METRICS revenue.total_revenue, revenue.avg_daily_revenue
)
ORDER BY time_period
LIMIT 6
""").to_pandas()

print("=== MONTHLY (default) ===")
print(df.to_string(index=False))

In [ ]:
# Quarterly — same query, just change the variable
df = session.sql("""
SELECT * FROM SEMANTIC_VIEW(
  sv_variable_time_grain
  DIMENSIONS revenue.time_period
  METRICS revenue.total_revenue, revenue.avg_daily_revenue
  VARIABLES time_grain = 'quarter'
)
ORDER BY time_period
""").to_pandas()

print("=== QUARTERLY ===")
print(df.to_string(index=False))

In [ ]:
# Yearly — same query, different variable
df = session.sql("""
SELECT * FROM SEMANTIC_VIEW(
  sv_variable_time_grain
  DIMENSIONS revenue.time_period
  METRICS revenue.total_revenue, revenue.avg_daily_revenue
  VARIABLES time_grain = 'year'
)
ORDER BY time_period
""").to_pandas()

print("=== YEARLY ===")
print(df.to_string(index=False))

In [ ]:
# Variables work with additional dimensions too
df = session.sql("""
SELECT * FROM SEMANTIC_VIEW(
  sv_variable_time_grain
  DIMENSIONS revenue.time_period, revenue.product_category
  METRICS revenue.total_revenue, revenue.total_units
  VARIABLES time_grain = 'quarter'
)
ORDER BY time_period, product_category
LIMIT 12
""").to_pandas()

print("=== QUARTERLY BY PRODUCT CATEGORY ===")
print(df.to_string(index=False))

### Pattern 1 Takeaways

| Pros | Cons |
|------|------|
| Single semantic view, one dimension | Cannot be combined with window function metrics (see Pattern 3) |
| Default value means queries work without specifying the variable | Variable expansion creates subqueries internally |
| Clean query syntax — just add `VARIABLES time_grain = 'quarter'` | Limited to expressions supported by variables (no subqueries) |
| Works with any additional dimensions/filters | |

---
## Pattern 2: Window Function Metrics

Window function metrics let you define **time-relative calculations** directly in the semantic view: rolling averages, running totals, lag/lead comparisons.

### How it works

The semantic view `sv_window_metrics` defines:
- A base metric (`daily_revenue = SUM(revenue)`)
- Window function metrics that operate **on the base metric**:

```sql
METRICS (
    revenue.daily_revenue AS SUM(revenue),

    -- Rolling 7-day average
    revenue.rolling_7day_avg AS AVG(daily_revenue)
      OVER (PARTITION BY EXCLUDING revenue.sale_date
            ORDER BY revenue.sale_date
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW),

    -- Revenue 7 days ago
    revenue.revenue_7_days_ago AS LAG(daily_revenue, 7)
      OVER (PARTITION BY EXCLUDING revenue.sale_date
            ORDER BY revenue.sale_date)
)
```

**Key concept: `PARTITION BY EXCLUDING`** — This removes the specified dimension from the partition. Whatever other dimensions appear in the query remain in the partition. This means the window adapts to whatever grouping the user queries with.

In [ ]:
# Rolling 7-day average and week-over-week comparison
df = session.sql("""
SELECT * FROM SEMANTIC_VIEW(
  sv_window_metrics
  DIMENSIONS revenue.sale_date
  METRICS revenue.daily_revenue, revenue.rolling_7day_avg, revenue.revenue_7_days_ago
)
WHERE sale_date >= '2024-06-01' AND sale_date <= '2024-06-14'
ORDER BY sale_date
""").to_pandas()

print("=== ROLLING 7-DAY AVG + WEEK-OVER-WEEK ===")
print(df.to_string(index=False))

In [ ]:
# Window metrics adapt when you add dimensions — here, partitioned by region
df = session.sql("""
SELECT * FROM SEMANTIC_VIEW(
  sv_window_metrics
  DIMENSIONS revenue.sale_date, revenue.region
  METRICS revenue.daily_revenue, revenue.rolling_7day_avg
  WHERE region = 'East'
)
WHERE sale_date >= '2024-06-01' AND sale_date <= '2024-06-07'
ORDER BY sale_date
""").to_pandas()

print("=== ROLLING AVG PARTITIONED BY REGION (East only) ===")
print(df.to_string(index=False))

In [ ]:
# 30-day rolling total
df = session.sql("""
SELECT * FROM SEMANTIC_VIEW(
  sv_window_metrics
  DIMENSIONS revenue.sale_date
  METRICS revenue.daily_revenue, revenue.rolling_30day_total, revenue.revenue_30_days_ago
)
WHERE sale_date >= '2024-03-01' AND sale_date <= '2024-03-10'
ORDER BY sale_date
""").to_pandas()

print("=== 30-DAY ROLLING TOTAL + 30-DAY LAG ===")
print(df.to_string(index=False))

### Pattern 2 Takeaways

| Pros | Cons |
|------|------|
| Powerful time-relative calculations (rolling, lag, cumulative) | Window frame bounds are fixed at view definition time |
| `PARTITION BY EXCLUDING` adapts to query dimensions automatically | Must include ORDER BY/EXCLUDING dimensions in every query |
| Multiple window metrics can reference the same base metric | Cannot use variables in frame bounds |
| Composable: LAG of an AVG, etc. | Window function metrics can't reference other window function metrics |

---
## Pattern 3: Combined — Multiple Grain Dimensions + Window Functions

This pattern combines the flexibility of different time grains with window function calculations. Since variables can't be used with window functions directly (see Limitations below), we define **separate dimensions for each grain** and let users choose which to query.

### How it works

The semantic view `sv_combined_dynamic` defines:
- Three grain dimensions: `month_period`, `quarter_period`, `year_period`
- Window functions tied to specific grains:

```sql
DIMENSIONS (
    revenue.month_period AS TO_CHAR(sale_date, 'YYYY-MM'),
    revenue.quarter_period AS TO_CHAR(sale_date, 'YYYY') || '-Q' || ...,
    revenue.year_period AS TO_CHAR(sale_date, 'YYYY'),
    ...
)
METRICS (
    revenue.period_revenue AS SUM(revenue),

    revenue.prior_month_revenue AS LAG(period_revenue, 1)
      OVER (PARTITION BY EXCLUDING revenue.month_period
            ORDER BY revenue.month_period),

    revenue.prior_quarter_revenue AS LAG(period_revenue, 1)
      OVER (PARTITION BY EXCLUDING revenue.quarter_period
            ORDER BY revenue.quarter_period),
    ...
)
```

Users pick which grain to query with, and get the corresponding period-over-period metrics.

In [ ]:
# Monthly grain: revenue + prior month + 3-month moving average
df = session.sql("""
SELECT * FROM SEMANTIC_VIEW(
  sv_combined_dynamic
  DIMENSIONS revenue.month_period
  METRICS revenue.period_revenue, revenue.prior_month_revenue, revenue.moving_avg_3_months
)
ORDER BY month_period
LIMIT 12
""").to_pandas()

print("=== MONTHLY: Revenue + Prior Month + 3-Month Moving Avg ===")
print(df.to_string(index=False))

In [ ]:
# Quarterly grain: revenue + prior quarter
df = session.sql("""
SELECT * FROM SEMANTIC_VIEW(
  sv_combined_dynamic
  DIMENSIONS revenue.quarter_period
  METRICS revenue.period_revenue, revenue.prior_quarter_revenue
)
ORDER BY quarter_period
""").to_pandas()

print("=== QUARTERLY: Revenue + Prior Quarter ===")
print(df.to_string(index=False))

In [ ]:
# Monthly with cumulative total
df = session.sql("""
SELECT * FROM SEMANTIC_VIEW(
  sv_combined_dynamic
  DIMENSIONS revenue.month_period
  METRICS revenue.period_revenue, revenue.cumulative_monthly
)
ORDER BY month_period
""").to_pandas()

print("=== MONTHLY: Revenue + Cumulative Total ===")
print(df.to_string(index=False))

In [ ]:
# Monthly with region breakdown — window partitions by region automatically
df = session.sql("""
SELECT * FROM SEMANTIC_VIEW(
  sv_combined_dynamic
  DIMENSIONS revenue.month_period, revenue.region
  METRICS revenue.period_revenue, revenue.prior_month_revenue
  WHERE region = 'East'
)
ORDER BY month_period
LIMIT 6
""").to_pandas()

print("=== MONTHLY BY REGION (East): Revenue + Prior Month ===")
print(df.to_string(index=False))

### Pattern 3 Takeaways

| Pros | Cons |
|------|------|
| Full power of window functions at any grain | More dimensions/metrics to define (one per grain) |
| Users pick grain by choosing which dimension to query | Not as clean as a single variable swap |
| Prior period, cumulative, moving average all adapt | Need to define parallel metrics for each grain (prior_month vs prior_quarter) |
| `PARTITION BY EXCLUDING` handles additional dimensions automatically | |

---
## Limitations & Workarounds

### Variables + Window Functions: Not Compatible

You **cannot** use a variable-driven dimension in a window function's PARTITION BY or ORDER BY clause. Snowflake expands variables into subqueries internally, and those subqueries aren't valid GROUP BY expressions when window functions are involved.

**This fails:**
```sql
-- ERROR: not a valid group by expression
CREATE SEMANTIC VIEW broken_example
  VARIABLES (time_grain VARCHAR DEFAULT 'month')
  DIMENSIONS (
    revenue.time_period AS CASE time_grain WHEN 'month' THEN ... END
  )
  METRICS (
    revenue.period_revenue AS SUM(revenue),
    revenue.prior_period AS LAG(period_revenue, 1)
      OVER (PARTITION BY EXCLUDING revenue.time_period
            ORDER BY revenue.time_period)  -- FAILS: time_period uses a variable
  )
```

**Workaround:** Use Pattern 3 (separate dimensions per grain) when you need window functions.

### Variables Cannot Be Used in Frame Bounds

You cannot parameterize the window frame size:
```sql
-- NOT SUPPORTED:
RANGE BETWEEN INTERVAL lookback_days || ' days' PRECEDING AND CURRENT ROW
```

**Workaround:** Define separate window function metrics for each frame size you need (7-day, 30-day, 90-day).

### Window Function Metrics Can't Reference Other Window Function Metrics

You can't chain window metrics (e.g., LAG of a rolling average). Each window function metric must reference a base metric or metric expression directly.

---
## Summary: Choosing the Right Pattern

| Pattern | Best For | Key Feature |
|---------|----------|-------------|
| **1: Variables** | Swappable time grain on simple aggregations (SUM, AVG, COUNT) | `VARIABLES time_grain = 'quarter'` at query time |
| **2: Window Functions** | Time-relative calcs (rolling avg, lag, cumulative) at fixed grain | `PARTITION BY EXCLUDING` + window frames |
| **3: Combined** | Period-over-period + movable grain (MoM, QoQ with moving averages) | Separate dimensions per grain + window metrics per grain |

### Decision Guide

- **"I just need total/average/count at different grains"** → Pattern 1 (Variables)
- **"I need rolling averages or lag comparisons at a fixed grain"** → Pattern 2 (Window Functions)
- **"I need period-over-period comparisons AND I want to switch grains"** → Pattern 3 (Combined)

### References

- [CREATE SEMANTIC VIEW: Window Function Metrics](https://docs.snowflake.com/en/sql-reference/sql/create-semantic-view#label-create-semantic-view-window-function)
- [Using Variables in a Semantic View](https://docs.snowflake.com/en/user-guide/views-semantic/variables)
- [Querying Semantic Views](https://docs.snowflake.com/en/user-guide/views-semantic/querying)

In [ ]:
# Optional: Clean up (uncomment to run)
# session.sql("DROP DATABASE IF EXISTS DYNAMIC_METRICS_DEMO CASCADE").collect()
# print("Lab resources cleaned up.")